Modelo LightGMB Data set original

In [1]:
%pip install pandas
%pip install nltk
%pip install scikit-learn
%pip install lightgbm
%pip install optuna
%pip install numpy
%pip install matplotlib
%pip install seaborn
!python -c "import nltk; nltk.download('punkt_tab')"



Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
  Using cached optuna-4.6.0-py3-none-any.whl.metadata (17 kB)
  Using cached optuna-4.6.0-py3-none-any.whl.metadata (17 kB)
  Using cached alembic-1.17.2-py3-none-any.whl.metadata (7.2 kB)
  Using cached alembic-1.17.2-py3-none-any.whl.metadata (7.2 kB)
  Using cached mako-1.3.10-py3-none-any.whl.metadata (2.9 kB)
  Using cached mako-1.3.10-py3-none-any.whl.metadata (2.9 kB)
Using cached optuna-4.6.0-py3-none-any.whl (404 kB)
Using cached alembic-1.17.2-py3-none-any.w

In [2]:
# =========================================================================
# I. IMPORTACIONES Y CONFIGURACIÓN
# =========================================================================
import pandas as pd
import numpy as np
import re
import sys
import warnings
import joblib 
import os     

# NLP Libraries
import spacy
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score, classification_report

# LightGBM y Optuna
import lightgbm as lgb
import optuna

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)

# --- Configuración Base ---
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
N_TRIALS_OPTUNA = 10 
FILEPATH = '../resources/dataset/youtoxic_english_1000.csv' # ¡REVISA ESTA RUTA!

# Configuración de Guardado
MODEL_PATH = '../resources/models'
MODEL_FILENAME = 'model-LightGBM-original.pkl'

# Etiquetas de clasificación
TARGET_COLS = [
    'IsToxic', 
    'IsAbusive', 
    'IsProvocative', 
    'IsObscene', 
    'IsHatespeech', 
    'IsRacist']

# --- Inicialización de NLP y Recursos ---
try:
    print("1.1. Inicializando Recursos NLP...")
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)
    nltk.download('stopwords', quiet=True)
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
    stemmer = PorterStemmer()
    stop_words_nltk = set(stopwords.words('english'))
    print("✅ Recursos NLP inicializados correctamente.")
except Exception as e:
    print(f"❌ ERROR al inicializar librerías NLP: {e}")
    sys.exit(1)


# =========================================================================
# II. PREPROCESAMIENTO DETALLADO (Pasos 2, 3 y 4)
# =========================================================================

def preprocess_text(text):
    """
    Aplica todos los pasos de preprocesamiento definidos (Tokenización, Normalización, POS Tagging).
    """
    
    # 3. Normalización (Expresiones Regulares y Minúsculas)
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    if not text:
        return ""

    # 2. Tokenización (usando NLTK)
    tokens = word_tokenize(text)
    final_tokens = []
    
    # Usamos SpaCy para el POS Tagging (es más robusto)
    doc = nlp(" ".join(tokens))
    
    for token in doc:
        word = token.text
        
        # Eliminación de Stop Words
        if word in stop_words_nltk:
            continue
        
        # Etiquetado Gramatical (POS Tagging)
        if token.is_alpha:
            
            # Stemming
            stemmed_word = stemmer.stem(word)
            final_tokens.append(stemmed_word)

    return " ".join(final_tokens)


# =========================================================================
# III. FUNCIONES DE APOYO (Optuna y Umbrales)
# =========================================================================

# --- Función de Optimización (LightGBM) ---
def objective_lgbm(trial, X_train, Y_multi_train, X_test, Y_multi_test):
    """Función objetivo de Optuna para LightGBM (F1 Micro)."""
    lgb_params = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'n_estimators': trial.suggest_int('n_estimators', 100, 300),
        'learning_rate': trial.suggest_float('learning_rate', 0.05, 0.15, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 50),
        'max_depth': trial.suggest_int('max_depth', 5, 12),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True),
        'scale_pos_weight': trial.suggest_int('scale_pos_weight', 5, 20),
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
        'verbose': -1
    }

    base_clf = lgb.LGBMClassifier(**lgb_params)
    model_ovr = OneVsRestClassifier(base_clf)
    
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        model_ovr.fit(X_train, Y_multi_train) 

    Y_pred = model_ovr.predict(X_test)
    f1_micro = f1_score(Y_multi_test, Y_pred, average='micro', zero_division=0)
    
    return f1_micro

# --- Función para Optimizar Umbrales ---
def optimize_thresholds(Y_multi_test, Y_proba_test, target_cols):
    """
    Busca el mejor umbral por etiqueta para maximizar el F1-Score en el conjunto de prueba.
    Devuelve un diccionario con los umbrales óptimos.
    """
    optimal_thresholds = {}
    
    print("\n   > Optimizando Umbrales...")
    
    for i, label in enumerate(target_cols):
        best_f1 = -1
        best_thresh = 0.5
        
        # Probamos umbrales de 0.05 a 0.95
        for thresh in np.arange(0.05, 0.96, 0.05):
            # Convierte las probabilidades a predicciones binarias usando el umbral
            Y_pred_thresh = (Y_proba_test[:, i] > thresh).astype(int)
            
            # Calcula el F1-Score para esta etiqueta y este umbral
            # Usamos la columna real del DataFrame para Y_multi_test[label]
            f1 = f1_score(Y_multi_test[label], Y_pred_thresh, zero_division=0)
            
            if f1 > best_f1:
                best_f1 = f1
                best_thresh = thresh
        
        optimal_thresholds[label] = best_thresh
        
    print("   ✓ Umbrales óptimos encontrados.")
    return optimal_thresholds

# =========================================================================
# IV. FUNCIÓN PRINCIPAL Y EJECUCIÓN
# =========================================================================

def main():
    """Ejecuta el pipeline completo de LightGBM."""
    
    # 1. Carga del Dataset (Paso 1)
    print("\n2. Carga del Dataset...")
    try:
        df = pd.read_csv(FILEPATH)
        df['Text'] = df['Text'].fillna('')
        
        for col in TARGET_COLS:
            if col in df.columns:
                if df[col].dtype != 'int64': 
                    df[col] = df[col].astype(int)
            else:
                print(f"❌ ERROR: Columna target '{col}' no encontrada en el CSV.")
                sys.exit(1)
        
        X = df['Text']
        Y_multi = df[TARGET_COLS]
        
        X_train_raw, X_test_raw, _, _ = train_test_split(
            X, Y_multi, test_size=0.2, random_state=RANDOM_STATE, stratify=df['IsToxic']
        )
        Y_multi_train = Y_multi.iloc[X_train_raw.index].reset_index(drop=True)
        Y_multi_test = Y_multi.iloc[X_test_raw.index].reset_index(drop=True)
        X_train_raw = X_train_raw.reset_index(drop=True)
        X_test_raw = X_test_raw.reset_index(drop=True)
        
        print(f"   ✓ Datos cargados. Train: {len(X_train_raw)}, Test: {len(X_test_raw)}")
        
    except FileNotFoundError:
        print(f"❌ ERROR CRÍTICO: ARCHIVO NO ENCONTRADO en la ruta: {FILEPATH}")
        print("💡 Por favor, ajusta la variable 'FILEPATH' con la ubicación correcta de tu CSV.")
        sys.exit(1)

    # 2. Preprocesamiento (Pasos 2, 3, 4)
    print("\n3. Preprocesamiento de Textos (Tokenización, Normalización, POS/Stemming)...")
    X_train_processed = X_train_raw.apply(preprocess_text)
    X_test_processed = X_test_raw.apply(preprocess_text)
    print("   ✓ Preprocesamiento finalizado.")
    
    # 3. Vectorización (Paso 5: TF-IDF)
    print("\n4. Vectorización TF-IDF (Paso 5)...")
    vectorizer = TfidfVectorizer(
        ngram_range=(1, 3), 
        stop_words=list(ENGLISH_STOP_WORDS), 
        max_df=0.8, 
        min_df=5,
        sublinear_tf=True
    )
    
    X_train_vec = vectorizer.fit_transform(X_train_processed)
    X_test_vec = vectorizer.transform(X_test_processed)
    print(f"   ✓ Dimensión de Features: {X_train_vec.shape[1]}")

    # 4. Entrenamiento y Optimización con Optuna
    print(f"\n5. Entrenamiento y Optimización LightGBM ({N_TRIALS_OPTUNA} trials)...")
    study = optuna.create_study(direction='maximize')
    study.optimize(
        lambda trial: objective_lgbm(trial, X_train_vec, Y_multi_train, X_test_vec, Y_multi_test), 
        n_trials=N_TRIALS_OPTUNA, 
        show_progress_bar=True
    )
    
    # Entrenamiento Final
    print("   ✓ Optuna finalizado. Entrenando modelo final...")
    best_params = study.best_params
    best_params.update({'objective': 'binary', 'metric': 'binary_logloss', 'random_state': RANDOM_STATE, 'n_jobs': -1, 'verbose': -1})

    model_lgbm = OneVsRestClassifier(lgb.LGBMClassifier(**best_params))
    model_lgbm.fit(X_train_vec, Y_multi_train) 
    print("   ✓ Entrenamiento del modelo final completado.")
    
    
    # 5. Evaluación, Overfitting y Guardado
    print("\n6. Evaluación y Guardado...")

    # Generar probabilidades para el ajuste de umbral
    Y_proba_train = model_lgbm.predict_proba(X_train_vec)
    Y_proba_test = model_lgbm.predict_proba(X_test_vec)

    # ----------------------------------------------------
    # A) EVALUACIÓN (BEFORE OPT) - Umbral 0.5 Fijo
    # ----------------------------------------------------
    y_train_pred_05 = model_lgbm.predict(X_train_vec)
    y_test_pred_05 = model_lgbm.predict(X_test_vec)
    
    f1_train_05 = f1_score(Y_multi_train, y_train_pred_05, average='micro', zero_division=0)
    f1_test_05 = f1_score(Y_multi_test, y_test_pred_05, average='micro', zero_division=0)
    overfitting_05 = f1_train_05 - f1_test_05
    
    print("\n--- A) RENDIMIENTO (BEFORE OPT) - UMBRAL 0.5 ---")
    print(f"Micro F1-Score (Train): {f1_train_05:.4f}")
    print(f"Micro F1-Score (Test): {f1_test_05:.4f}")
    print(f"⚠️  Overfitting (Train F1 - Test F1): {overfitting_05:.4f}")
    
    # ----------------------------------------------------
    # B) EVALUACIÓN (AFTER OPT) - Umbral Optimizado
    # ----------------------------------------------------
    
    # 1. Optimizar los umbrales
    optimal_thresholds = optimize_thresholds(Y_multi_test, Y_proba_test, TARGET_COLS)
    
    # 2. Aplicar umbrales óptimos en Test
    # Convertimos las probabilidades a binario usando los umbrales específicos
    threshold_values = np.array(list(optimal_thresholds.values()))
    Y_pred_test_opt = (Y_proba_test > threshold_values).astype(int)
    f1_test_opt = f1_score(Y_multi_test, Y_pred_test_opt, average='micro', zero_division=0)

    # 3. Aplicar umbrales óptimos en Train para calcular Overfitting 'After Opt'
    Y_pred_train_opt = (Y_proba_train > threshold_values).astype(int)
    f1_train_opt = f1_score(Y_multi_train, Y_pred_train_opt, average='micro', zero_division=0)

    # 4. Cálculo final de Overfitting (After Opt)
    overfitting_opt = f1_train_opt - f1_test_opt

    print("\n--- B) RESULTADOS FINALES (AFTER OPT) - UMBRAL OPTIMIZADO ---")
    print(f"Micro F1-Score (Train): {f1_train_opt:.4f}")
    print(f"Micro F1-Score (Test - After Opt): {f1_test_opt:.4f}")
    print(f"✅ Overfitting (After Opt): {overfitting_opt:.4f}")

    print("\nUmbrales Óptimos por Etiqueta (Test):")
    # Imprimimos los umbrales óptimos encontrados
    for label, thresh in optimal_thresholds.items():
        print(f"   {label}: {thresh:.2f}")

    # Informe de Clasificación Detallado (Usando Umbral Optimizado)
    print("\nInforme de Clasificación Detallado (Umbral Optimizado):")
    print(classification_report(Y_multi_test, Y_pred_test_opt, target_names=TARGET_COLS, zero_division=0))
    
    # c) Guardado del Modelo
    os.makedirs(MODEL_PATH, exist_ok=True)
    joblib.dump(model_lgbm, f"{MODEL_PATH}/{MODEL_FILENAME}")
    print(f"\n💾 Modelo guardado exitosamente en: {MODEL_PATH}/{MODEL_FILENAME}")
    
    # d) Guardado del Vectorizador
    vectorizer_filename = 'lightgbm_tfidf_vectorizer.pkl'
    joblib.dump(vectorizer, f"{MODEL_PATH}/{vectorizer_filename}")
    print(f"💾 Vectorizador guardado exitosamente en: {MODEL_PATH}/{vectorizer_filename}")

if __name__ == "__main__":
    main()

1.1. Inicializando Recursos NLP...
✅ Recursos NLP inicializados correctamente.

2. Carga del Dataset...
   ✓ Datos cargados. Train: 800, Test: 200

3. Preprocesamiento de Textos (Tokenización, Normalización, POS/Stemming)...
✅ Recursos NLP inicializados correctamente.

2. Carga del Dataset...
   ✓ Datos cargados. Train: 800, Test: 200

3. Preprocesamiento de Textos (Tokenización, Normalización, POS/Stemming)...
   ✓ Preprocesamiento finalizado.

4. Vectorización TF-IDF (Paso 5)...
   ✓ Dimensión de Features: 571

5. Entrenamiento y Optimización LightGBM (10 trials)...
   ✓ Preprocesamiento finalizado.

4. Vectorización TF-IDF (Paso 5)...
   ✓ Dimensión de Features: 571

5. Entrenamiento y Optimización LightGBM (10 trials)...


Best trial: 9. Best value: 0.568452: 100%|██████████| 10/10 [00:11<00:00,  1.18s/it]



   ✓ Optuna finalizado. Entrenando modelo final...
   ✓ Entrenamiento del modelo final completado.

6. Evaluación y Guardado...

--- A) RENDIMIENTO (BEFORE OPT) - UMBRAL 0.5 ---
Micro F1-Score (Train): 0.7306
Micro F1-Score (Test): 0.5685
⚠️  Overfitting (Train F1 - Test F1): 0.1622

   > Optimizando Umbrales...
   ✓ Entrenamiento del modelo final completado.

6. Evaluación y Guardado...

--- A) RENDIMIENTO (BEFORE OPT) - UMBRAL 0.5 ---
Micro F1-Score (Train): 0.7306
Micro F1-Score (Test): 0.5685
⚠️  Overfitting (Train F1 - Test F1): 0.1622

   > Optimizando Umbrales...
   ✓ Umbrales óptimos encontrados.

--- B) RESULTADOS FINALES (AFTER OPT) - UMBRAL OPTIMIZADO ---
Micro F1-Score (Train): 0.7217
Micro F1-Score (Test - After Opt): 0.5847
✅ Overfitting (After Opt): 0.1370

Umbrales Óptimos por Etiqueta (Test):
   IsToxic: 0.45
   IsAbusive: 0.50
   IsProvocative: 0.45
   IsObscene: 0.45
   IsHatespeech: 0.60
   IsRacist: 0.70

Informe de Clasificación Detallado (Umbral Optimizado):
    